In [84]:
from pathlib import Path
import polars as pl

In [79]:
raw_dir = Path('../data/raw')
bronze_dir = Path('../data/bronze/buildings')

### **Check file count & file name**

In [82]:
raw_files = sorted(raw_dir.glob('*.csv.gz'))
bronze_files = sorted(bronze_dir.glob('*.parquet'))

raw_expected_bronze_names = {raw_file.name.replace('.csv.gz', '.parquet') for raw_file in raw_files}
actual_bronze_names = {bronze_file.name for bronze_file in bronze_files}

missing_bronze = sorted(raw_expected_bronze_names - actual_bronze_names)
unexpected_bronze = sorted(actual_bronze_names - raw_expected_bronze_names)

counts_match = 'Yes' if len(raw_files) == len(bronze_files) else 'No'
mapping_match = 'Yes' if not missing_bronze and not unexpected_bronze else 'No'

print(f'Raw file count: {len(raw_files)}')
print(f'Bronze file count: {len(bronze_files)}')
print(f'Counts match: {counts_match}')
print(f'One-to-one filename mapping: {mapping_match}')

Raw file count: 601
Bronze file count: 601
Counts match: Yes
One-to-one filename mapping: Yes


### **Metadata check**

In [93]:
required_cols = {'country', 'quadkey', 'upload_date', 'source_file'}

all_ok = True

for file in bronze_files:
    lf = pl.scan_parquet(file)
    cols = set(lf.collect_schema().names())
    missing = required_cols - cols

    if missing:
        print("missing", missing, "in", file)
        all_ok = False

if all_ok:
    print("OK - all files passed")

OK - all files passed
